# Chat loop and token budget

In this notebook, we will:
1. Compare API requests with and without conversation history.
2. Repeat the same prompt and compare the answers.
3. Count tokens and estimate input costs for both chains.
4. Compare API usage with low and high reasoning effort.

**Data:** uses `data/ikea_reviews.csv` and `data/mcdonalds_reviews.csv`, unpacked once with [explore_yelp_data.ipynb](../01_Explore_Yelp_Data/explore_yelp_data.ipynb). Reuse the files; no new data preparation is needed.


## Libraries and settings

`client.responses.create()` of the OpenAI library sends one request to the model and returns its answer. Every call is a new request: running a cell again sends its requests again and can give different answers.

In [ ]:
# Libraries
import os                                                       #Bibliothek für Zugriff auf Betriebssystemfunktionen, z.B. aktuellea Arbeitsverzeichnis
import platform                                                 #Bibliothek zum Abrufen von Informationen über das Betriebssystem
from datetime import datetime                                   #Klasse datetime importieren, um mit Datum und Uhrzeit zu arbeiten
from pathlib import Path                                        #Klasse Path importieren, um Ordner- und Dateipfade zu verwalten
from platform import python_version                             #Funktion importieren, um die installierte Python-Version abzufragen

import pandas as pd                                             #Bibliothek Pandas für die Verarbeitung von Tabellen importieren, unter dem Kurznamen pd
import tiktoken                                                 #Bibliothek zum Zählen von Tokens importieren, also Texteinheiten, die Sprachmodelle verarbeiten
from dotenv import load_dotenv                                  #Funktion importieren, um Umgebungsvariablen wie API-Schlüssel aus einer .env-Datei zu laden
from openai import OpenAI                                       #Klasse OpenAI importieren, um einen Client für die Kommunikation mit der OpenAI-API zu erstellen

# Folder with the data (relative to the folder of this notebook)
DATA = Path("../data")                                          #Variable DATA speichert Pfad zum Ordner data, Path erstellt daraus ein Pfadobjekt

# Model and course price assumption in USD per 1 million input tokens
MODEL = "gpt-6-luna"                                            #Variable MODEL speichert Modellnamen, damit er später mit model=MODEL an die API übergeben werden kann
PRICE_INPUT_PER_M = 0.10                                        #Variable speichert den angenommenen Preis von 0.10 USD pro 1 Million Eingabetokens

# The API key is a Codespaces secret; outside Codespaces it is read from .env
load_dotenv("../.env")                                          #Datei .env im übergeordneten Ordner lesen und ihre Umgebungsvariablen laden, z.B. den geheimen API-Schlüssel

# The client sends our requests and retries when the rate limit is reached
client = OpenAI(max_retries=8)                                  #OpenAI() erstellt einen API-Client und speichert ihn in client. Dieser verwendet den geladenen API-Schlüssel, sendet später unsere Anfragen und wiederholt fehlgeschlagene Anfragen bei bestimmten Fehlern bis zu 8-mal

# Show current working directory
print(os.getcwd())                                              #os.getcwd() ermittelt den vollständigen Pfad des aktuellen Arbeitsverzeichnisses, print() zeigt ihn an


/workspaces/llm_fundamentals/02_Chat_Loop_Tokens


## Step 1: Ask the API a follow-up question

We send two separate requests. Does the model understand what "it" means in the second question?

In [ ]:
# First question
r1 = client.responses.create(                                                   #Erste Anfrage an das Modell senden und die Antwort in r1 speichern. #client ist die Verbindung zur OpenAI-API, responses ist der Bereich für Modellantworten, create() sendet eine neue Anfrage und speichert deren Antwort in r1
    model=MODEL,                                                                #Festgelegtes Sprachmodell verwenden. Variable MODEL = gpt-6-luna
    input=(                                                                     #Frage und Anweisung als Eingabetext übergeben
        "Name one typical dish in the IKEA restaurant. "
        "Answer in one sentence."
    ),
)
print("Answer 1:", r1.output_text)                                              #Text der ersten Modellantwort aus r1 ausgeben, output_text ist Eigenschaftsname von OpenAI-Bibliothek

# Follow-up question, sent as a separate request
r2 = client.responses.create(model=MODEL, input="Why is it so popular?")        #Zweite unabhängige Anfrage senden, ohne den Kontext der ersten Anfrage
print("Answer 2:", r2.output_text)                                              #Text der zweiten Modellantwort aus r2 ausgeben


Answer 1: A typical dish at the IKEA restaurant is Swedish meatballs served with mashed potatoes and lingonberry sauce.
Answer 2: What are you referring to by “it”? Tell me the product, trend, or idea, and I’ll explain why it’s popular.


The second request contains no earlier question. The model may guess a plausible meaning of "it", but it has not seen the previous request. **These requests are independent: our code has not supplied any history.**

## Step 2: Keep a list of messages and send it again

A chat application like ChatGPT keeps the whole conversation in a list and sends the full list with every new question. Each message has a `role` (`system`, `user` or `assistant`) and a `content`.

In [ ]:
# The system message holds the rules for the whole conversation
RULES = "You answer questions about IKEA. Keep every answer to one sentence."   #Regeln als Text in der Variable RULES speichern
messages = [{"role": "system", "content": RULES}]  #Liste mit einem Dictionary erstellen: "role" und "content" sind von der API vorgegebene Schlüssel (Strings). "system" ist der Wert für "role" und bedeutet, dass die Nachricht Regeln enthält. RULES ist der Wert für "content" und liefert den zuvor gespeicherten Regeltext
# The questions of our conversation
conversation = [                                                                #Liste conversation mit drei Fragen erstellen
    "Name one typical dish in the IKEA restaurant.",
    "Why is it so popular?",
    "What was my first question?",
]

for question in conversation:                                                   #Jede Frage aus conversation nacheinander durchlaufen und jeweils in question speichern
    # 1. Add the question to the list
    messages.append({"role": "user", "content": question})                      #Aktuelle Frage als neue Benutzernachricht zur Liste messages hinzufügen

    # 2. Send the whole list
    r = client.responses.create(model=MODEL, input=messages)                    #Über client eine neue API-Anfrage erstellen, MODEL als Sprachmodell verwenden, alle bisherigen Nachrichten aus messages mitsenden und Antwort in r speichern

    # 3. Add the answer to the list
    messages.append({"role": "assistant", "content": r.output_text})            #Antworttext aus r als neue Assistentennachricht zu messages hinzufügen

    print("Q:", question)                                                       #Aktuelle Frage mit Q: ausgeben
    print("A:", r.output_text)                                                  #Antworttext des Modells mit A: ausgeben
    print("   input tokens of this request:", r.usage.input_tokens, "\n")       #Anzahl der Eingabetokens dieser Anfrage ausgeben, \n erzeugt Leerzeile

Q: Name one typical dish in the IKEA restaurant.
A: Swedish meatballs with lingonberry sauce are a typical IKEA restaurant dish.
   input tokens of this request: 32 

Q: Why is it so popular?
A: IKEA’s Swedish meatballs are popular for their comforting flavor, affordable price, and association with the store’s Swedish heritage.
   input tokens of this request: 64 

Q: What was my first question?
A: You asked, “Name one typical dish in the IKEA restaurant.”
   input tokens of this request: 105 



Now the model can answer the follow-up questions, because the earlier questions and answers are part of the input. The price: the input grows with every turn, so every turn costs more input tokens. Applications therefore often send only the last few turns, or replace older turns with a short summary. This limits the input tokens, but earlier details can get lost.

## Step 3: Same prompt, different answers

The model picks the next token with weighted dice. So the same prompt can give different answers. We send the same request five times.

In [ ]:
# The same prompt, five times
for i in [1, 2, 3, 4, 5]:                           #Schleife fünfmal durchlaufen, i enthält dabei nacheinander die Zahlen 1 bis 5
    r = client.responses.create(                    #Über den OpenAI-Client eine neue Anfrage an das Sprachmodell senden und die Antwort in r speichern
        model=MODEL, input="Describe IKEA in exactly five words."       #Modell aus der Variable MODEL verwenden und den Auftrag über input als Text übergeben
    )
    print("Run", i, ":", r.output_text)                                 #Nummer des aktuellen Durchlaufs und den Antworttext des Modells ausgeben


Run 1 : Affordable furniture, Swedish design, self-assembly.
Run 2 : Affordable flat-pack home furnishings worldwide.
Run 3 : Swedish-designed, affordable flat-pack home furnishings.
Run 4 : Affordable furniture with maze-like stores.
Run 5 : Swedish-inspired, affordable flat-pack home furnishings.


## Step 4: What does it cost to read all IKEA and McDonald's reviews?

We want to send every review to the model with this prompt: *"Positive, negative or mixed? Answer with one word."*

Guess the cost **before** you compute anything.

In [ ]:
# Your guess in USD (before you compute anything!)
my_guess_usd = None  # TODO: e.g. 0.50                #Schätzen, wie viel es in USD kostet, alle IKEA- und McDonald’s-Bewertungen durch das Sprachmodell analysieren zu lassen
my_guess_usd = 0.50

### What is a token?

A model does not read letters or words, but **tokens**: whole words, parts of words or punctuation marks. The price of a request depends on the number of tokens. `tiktoken` is the tokenizer of the OpenAI models.

In [ ]:
# Load the tokenizer
enc = tiktoken.get_encoding("o200k_base")                           #Tokenizer mit dem Namen o200k_base laden und in enc speichern. Er zerlegt Text in Tokens und ordnet jedem Token eine Zahlen-ID zu

# A review from IKEA Philadelphia (1 of 5 stars, original spelling)
review = "Swedish meatballs good. Attitude at.Return/Exchange counter bad."     #Bewertungstext als String in der Variable review speichern

# Text -> token IDs (numbers)
token_ids = enc.encode(review)                  #Bewertungstext mit dem Tokenizer enc in Tokens zerlegen und deren Zahlen-IDs als Liste token_ids speichern
print("Words: ", len(review.split()))           #Bewertungstext bei Leerzeichen aufteilen und die Anzahl der entstandenen Wörter ausgeben
print("Tokens:", len(token_ids))                #Anzahl der Token-IDs in token_ids zählen und ausgeben
print("IDs:   ", token_ids)                     #Liste aller Zahlen-IDs der Tokens ausgeben

# Token IDs -> pieces of text, one token at a time
pieces = []                                     #Leere Liste pieces erstellen, in der später die einzelnen Textstücke gespeichert werden
for token_id in token_ids:                      #Alle Zahlen-IDs aus token_ids nacheinander durchlaufen und die aktuelle ID in token_id speichern
    pieces.append(enc.decode([token_id]))       #Aktuelle Token-ID mit enc.decode() zurück in Text umwandeln und diesen Text zur Liste pieces hinzufügen
print("Pieces:", pieces)                        #Alle Textstücke aus pieces ausgeben, um zu sehen, wie der Tokenizer die Bewertung aufgeteilt hat


Words:  7
Tokens: 15
IDs:    [15254, 45708, 20872, 81079, 1899, 13, 11278, 5962, 540, 90846, 14, 35771, 9446, 4790, 13]
Pieces: ['Sw', 'edish', ' meat', 'balls', ' good', '.', ' Att', 'itude', ' at', '.Return', '/', 'Exchange', ' counter', ' bad', '.']


**Your turn:** try a German sentence, a very long word and a text with typos. What happens to the number of tokens?

In [ ]:
# TODO: replace the text and run the cell again
my_text = """Bei IKEA haben wir gefunden, was wir gesucht haben."""             #Eigenen Beispieltext als String in my_text speichern

# Text -> token IDs
token_ids = enc.encode(my_text)                     #Text aus my_text mit Tokenizer zerlegen und deren Zahlen-IDs in token_ids speichern
print("Words: ", len(my_text.split()))              #Text bei Leerzeichen aufteilen, die entstandenen Wörter zählen und deren Anzahl ausgeben
print("Tokens:", len(token_ids))                    #Anzahl der Tokens in token_ids zählen und ausgeben

# Token IDs -> pieces of text
pieces = []                                         #Leere Liste erstellen, um darin die einzelnen Textstücke zu sammeln
for token_id in token_ids:                          #Jede Token-ID aus token_ids nacheinander durchlaufen und in token_id speichern
    pieces.append(enc.decode([token_id]))           #Aktuelle Zahlen-ID zurück in das zugehörige Textstück umwandeln und an die Liste pieces anhängen
print("Pieces:", pieces)                            #Liste mit allen Textstücken ausgeben, um die Aufteilung des ursprünglichen Textes zu sehen


Words:  9
Tokens: 11
Pieces: ['Bei', ' IKEA', ' haben', ' wir', ' gefunden', ',', ' was', ' wir', ' gesucht', ' haben', '.']


### Count the tokens of all IKEA and McDonald's reviews

Counting happens on this computer. No review is sent to the API in this step, so we can use the original file.

In [ ]:
# Read both complete datasets; counting is local, nothing is sent to the API
ikea = pd.read_csv(DATA / "ikea_reviews.csv")
mcdonalds = pd.read_csv(DATA / "mcdonalds_reviews.csv")
ikea["chain"] = "IKEA"
mcdonalds["chain"] = "McDonald's"
df = pd.concat([ikea, mcdonalds], ignore_index=True)

# Count the tokens of every review
tokens = []
for text in df["text"]:
    token_ids = enc.encode(text)
    tokens.append(len(token_ids))
df["tokens"] = tokens

# Reviews and text tokens per chain
budget = df.groupby("chain").agg(
    reviews=("text", "size"), text_tokens=("tokens", "sum")
)
budget

### Compute the cost and compare it with your guess

In [ ]:
# Every review gets its own prompt
PROMPT = "Positive, negative or mixed? Answer with one word.\n"
prompt_tokens = len(enc.encode(PROMPT))

# Input tokens: review tokens plus one prompt per review
budget["input_tokens"] = (
    budget["text_tokens"] + budget["reviews"] * prompt_tokens
)

# Cost: tokens / 1 million x price per 1 million tokens
budget["input_cost_usd"] = (
    budget["input_tokens"] / 1_000_000 * PRICE_INPUT_PER_M
)
cost = budget["input_cost_usd"].sum()

print(budget.to_string())
print(f"\nBoth chains (input): {cost:.4f} USD")
print(f"Your guess:          {my_guess_usd} USD")

The larger chain has more reviews, but review length also affects its token budget. These counts include review text and the repeated prompt; API message formatting adds overhead.

The tokens of the answers (output tokens) come on top; they are more expensive than input tokens.

### Context window

The context window is the maximum number of tokens in one request: input plus output. For this exercise, use a context budget of 1,000,000 tokens. Reserve space for the prompt, message formatting and output as well as the review texts.

## Step 5: Usage and reasoning effort

Send the same short review with low and high reasoning effort. Compare the label, input tokens and output tokens with the local count. The usage also reports how many output tokens were used for hidden reasoning.

In [ ]:
# Local token count of the same input, for comparison with the API usage
print("Local input-token count:", len(enc.encode(PROMPT + review)))

# Send the same input with low and high reasoning effort.
# store=False: the provider does not keep the request.
for effort in ["low", "high"]:
    r = client.responses.create(
        model=MODEL,
        input=PROMPT + review,
        reasoning={"effort": effort},
        store=False,
    )
    usage = r.usage
    print("Effort:", effort, "| answer:", r.output_text)
    print(
        "Input:",
        usage.input_tokens,
        "| output:",
        usage.output_tokens,
        "| of these, reasoning:",
        usage.output_tokens_details.reasoning_tokens,
    )

The API count includes message formatting. Its output-token count can include hidden reasoning tokens even when the visible answer is one word. Higher reasoning effort can increase output tokens and cost; a useful quality improvement must be measured.

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. Why can the model refer to an earlier question in step 2? Does that mean its weights changed during the conversation?
2. Use one tokenization output to explain why counting words is not enough to estimate the input cost.
3. Why must the cost calculation count the prompt once per review? Which costs are still missing from the input-only estimate?
4. Why can repeated requests give different answers? Why would one run be weak evidence when comparing prompts?
5. How can a one-word answer use more than one output token? How could you decide whether higher reasoning effort is worth its cost?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. The application resends the message history. The model sees it in the current input; its weights do not change.

2. Tokens can be words, word pieces or punctuation. Their number depends on the text and tokenizer; refer to your measured pieces.

3. Each separate request contains its own prompt. Output tokens, including hidden reasoning tokens, cost extra; message formatting also adds input tokens.

4. Sampling can choose different next tokens. Compare several runs on the same inputs with the same model and settings.

5. The usage can include hidden reasoning tokens and answer punctuation. Compare measured correctness, output tokens and response time on several examples.

</details>

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print("-----------------------------------")
print(os.name.upper())
print(platform.system(), "|", platform.release())
print("Datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python Version:", python_version())
print("-----------------------------------")
